# Alternative-scaled multinomial logit

Estimate an MNL model with alternative-specific utility scales. The SM scale is fixed to one for identification.

This notebook is self-contained and was executed in the repository's Office
validation environment. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import numpy as np
import pandas as pd
import torch

from torchdcm import (
    AlternativeScale,
    Beta,
    ChoiceDataset,
    ChoiceLatentEffect,
    ContinuousIndicator,
    CovariateScale,
    CovariateScaledMultinomialLogit,
    CrossNest,
    CrossNestedLogit,
    ErrorComponent,
    ErrorComponentsLogit,
    HybridChoiceModel,
    LatentClassLogit,
    LatentVariable,
    MixedLogit,
    MultinomialLogit,
    Nest,
    NestedLogit,
    OrderedChoiceDataset,
    OrderedLogit,
    OrderedProbit,
    RandomCoefficient,
    ScaledMultinomialLogit,
    UtilitySpec,
    WTPCoefficient,
    WTPMixedLogit,
)
from torchdcm.datasets import make_swissmetro_like

torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


def show_result(result):
    print(f"Final log likelihood: {result.loglike:.6f}")
    print(f"AIC: {result.aic:.3f}; BIC: {result.bic:.3f}")
    return pd.DataFrame(
        {
            "estimate": result.values,
            "std. error": result.bse,
            "z": result.zvalues,
            "p-value": result.pvalues,
        },
        index=pd.Index(result.param_names, name="parameter"),
    ).round(6)


TorchDCM example running on cuda


In [2]:
def make_choice_data(n_obs=180, seed=7, *, observation_variables=None):
    frame = make_swissmetro_like(n_obs=n_obs, seed=seed)
    data = ChoiceDataset.from_wide(
        frame,
        alternatives=["TRAIN", "SM", "CAR"],
        choice="choice",
        variables={
            "time": {
                "TRAIN": "time_train",
                "SM": "time_sm",
                "CAR": "time_car",
            },
            "cost": {
                "TRAIN": "cost_train",
                "SM": "cost_sm",
                "CAR": "cost_car",
            },
        },
        obs_variables=observation_variables,
        availability={
            "TRAIN": "avail_train",
            "SM": "avail_sm",
            "CAR": "avail_car",
        },
        individual_id="person_id",
    )
    return frame, data


def make_utility_spec(suffix=""):
    tag = f"_{suffix}" if suffix else ""
    spec = UtilitySpec()
    spec.utility(
        "TRAIN",
        Beta(f"ASC_TRAIN{tag}", init=0.0)
        + Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    spec.utility(
        "SM",
        Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    spec.utility(
        "CAR",
        Beta(f"ASC_CAR{tag}", init=0.0)
        + Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    return spec


frame, data = make_choice_data(seed=16)
spec = make_utility_spec()

In [3]:
scales = {
    "TRAIN": AlternativeScale(init=0.90, name="SCALE_TRAIN"),
    "SM": AlternativeScale(init=1.00, fixed=True, name="SCALE_SM"),
    "CAR": AlternativeScale(init=1.10, name="SCALE_CAR"),
}
model = ScaledMultinomialLogit(
    spec,
    scales,
    device=device,
    max_iter=60,
)
result = model.fit(data)
show_result(result)


Final log likelihood: -172.408472
AIC: 356.817; BIC: 375.975


,estimate,std. error,z,p-value
parameter,,,,
ASC_TRAIN,0.210117,0.355779,0.590584,0.554799
B_TIME,-0.034916,0.014032,-2.488304,0.012835
B_COST,-0.157642,0.061273,-2.572796,0.010088
ASC_CAR,0.169559,0.467375,0.362790,0.716762
SCALE_TRAIN,0.963396,0.084942,11.341786,0.000000
SCALE_CAR,1.078914,0.094453,11.422801,0.000000
